Find the ventilation rate neccecary to limit the probability of infection (as calculated by CAiMIRA) below a predefined threashold.
Variability between model runs may cause the resulting probability of infection to sometimes be slightly higher than lim_probability_infection.

In [ ]:
from caimira.ventilation.scenarios import acl_1, acl_2, acl_3, acl_4
import caimira.ventilation.find_requirements as find_requirements
import caimira.ventilation.show_requirements as show_requirements
import caimira.ventilation.get_models as get_models
import caimira.ventilation.model_response as model_response

In [ ]:
mask_types = ['No mask', 'Cloth', 'Type I', 'FFP2']
i = 3

scenario, lim_probability_infection, sceanrio_name = acl_4(mask_infected=mask_types[i])

save_name = sceanrio_name.replace(" ", "").replace(",", "").replace("-", "").replace(":", "")

Find constant air exchange.

In [ ]:
show_requirements.plot_probabilities(
    scenario, 
    lim_probability_infection_list=[lim_probability_infection], 
    air_exch_list = list(range(0, 100, 2)),
    title=sceanrio_name, 
    plot_dose=False, 
    save_to=f"./results/pi_{save_name}.png",
    both_axis=True
)

Evaluate constant air exchange.

In [ ]:
const_air_exch = [15.] # Determine by inspecting graph above (more accurate results)
exposure_model = get_models.get_exposure_model(const_air_exch, model_response.first_vent_transition_times(scenario), scenario).build_model(1)

clean_air_per_sec_per_pers = find_requirements.clean_air_per_sec_per_pers(const_air_exch, exposure_model)
print(clean_air_per_sec_per_pers)
print()

ci = show_requirements.confidence_interval(const_air_exch, model_response.first_vent_transition_times(scenario), scenario)
print(const_air_exch)
print(f"[{ci[0]:.4f}, {ci[1]:.4f}]")
print()

show_requirements.plot_model_concentration_results(
    scenario, 
    const_air_exch, 
    title=sceanrio_name, 
    deterministic_CO2=True, 
    plot_clean_air_delivery=True, 
    save_to=f"./results/conc_const_vent_{save_name}.png"
)

Find and evaluate CO2 limit.

In [ ]:
t = 1 # Any time within occupancy since occupancy and ventilation constant
CO2_models = get_models.get_deterministic_CO2_models(exposure_model)
limit = find_requirements.concentration_limit(CO2_models, t, const_air_exch[0])
print(f"{limit:.0f}")

air_exch_l, vent_transition_t = find_requirements.find_next_air_exch_by_co2(
    scenario,
    air_exch_list=[0.25],                                     # Default no ventilation
    vent_transition_times=None,                               
    max_CO2=limit,                                            # Define a reasonable limit from the above plot 
    min_CO2_fraction=0.9,                                     # Determines the lower limit for the CO2 concentration. Ventilation is decreased when reaching this point
    target_CO2_fraction=0.99,
    max_ventilation_changes=5
)
ci = show_requirements.confidence_interval(air_exch_l, vent_transition_t, scenario)
print(f"[{ci[0]:.4f}, {ci[1]:.4f}]")
print()

show_requirements.plot_model_concentration_results(
    scenario, 
    air_exch_l, 
    vent_transition_t, 
    itle=sceanrio_name, 
    deterministic_CO2=True, 
    save_to=f"./results/conc_CO2_vent_{save_name}.png"
)